In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import timedelta

from config import *
from darts_models import *
from seasonal_drift_model import *
from SIR_EAKF_model import *

In [ ]:
df_edv = read_influenza_ed_visits_prop_data(data_dir, loc_name2abbr)
states = df_edv.columns[3:]
num_states = len(states)
locations = locations.loc[states,]
df_edv[states] = np.round(df_edv[states].mul(pop_per_loc_abbr.loc[states], axis=1) * 1e-3,0)
df_edv

In [ ]:
smooth_ili = False
regress_ili = True
df_ili = read_ili_incidence_data(data_dir, epiyear, epiweek, states, df_edv, 
                                     smooth=smooth_ili, scale=True, regress=regress_ili, plot=True)

df_edv = df_edv[df_edv.date<pd.to_datetime(ref_date)]
df_ili = df_ili[df_ili.date<pd.to_datetime(ref_date)]

plot_edv_with_ili = False
if(plot_edv_with_ili):
    fig, axs = plt.subplots(nrows=num_states, ncols=1, figsize=(7, 2 * num_states), sharex=True)
    for i, state in enumerate(states):
        ax = axs[i] if num_states > 1 else axs  # Handle single subplot case
        ax.plot(df_edv['date'], df_edv[state], label='ED Visits', color='blue')
        ax.plot(df_ili['date'], df_ili[state], label='Scaled ILI', color='red')
        ax.set_title(f"{state} - ED Visits vs. ILI (scaled)")
        ax.set_ylabel("Cases")
        ax.legend()
    plt.xlabel("Date")
    plt.tight_layout()
    plt.show()

In [ ]:
switch_epiyear = 2022
switch_epiweek = 26
switch_date = pd.to_datetime(epiweek_to_dates(switch_epiyear, switch_epiweek).enddate())

df1 = df_ili[df_ili.date < switch_date]
df2 = df_edv[(df_edv.date >= switch_date)]
df_hosp_ex = pd.concat([df1, df2])

hosp_ex_start_date = pd.to_datetime('2010-10-09', format="%Y-%m-%d")
df_hosp_ex = df_hosp_ex[df_hosp_ex['date']>=hosp_ex_start_date]
df_hosp_ex = df_hosp_ex.reset_index(drop=True)
df_hosp_ex[states] = df_hosp_ex[states].fillna(0)
# na_counts = df_hosp_ex[states].isna().sum()
# na_counts

covid_start_epiyear = 2020
covid_start_epiweek = 26
covid_start_date = pd.to_datetime(epiweek_to_dates(covid_start_epiyear, covid_start_epiweek).enddate())
covid_end_date = switch_date
df_hosp_ex = df_hosp_ex[~df_hosp_ex["date"].between(covid_start_date, covid_end_date, inclusive="left")]

df_hosp_ex_long = pd.melt(df_hosp_ex,id_vars=['date','year','week'],value_vars=df_hosp_ex.columns[3:],var_name='state',value_name='cases')
plot_hosp_ex = False
if(plot_hosp_ex):
    g = sns.FacetGrid(df_hosp_ex_long, col="state", col_wrap=5, hue="state", sharey=False, sharex=True, height=3, aspect=1.33)
    g.map(sns.lineplot, "date", "cases")
    [plt.setp(ax.get_xticklabels(), rotation=90) for ax in g.axes.flat]
    plt.subplots_adjust(hspace=0.4, wspace=0.4)

In [ ]:
past_weeks = 0 # Set this to produce past weeks forecasts
ref_dates = [pd.Timestamp(ref_date) + timedelta(weeks=w) for w in range(-past_weeks,1)]

# For reproducibility
np.random.seed(230098)
torch.manual_seed(42095703)
torch.set_float32_matmul_precision('medium')
darts_local_models, darts_global_models = get_darts_models(quantiles)

model_suffix = "_edv"
if(regress_ili):
    model_suffix = model_suffix + "_mod_ili"
if(smooth_ili):
    model_suffix = model_suffix + "_smooth"

for k in list(darts_local_models.keys()):
    darts_local_models[k + model_suffix] = darts_local_models.pop(k)

print(darts_local_models.keys())

In [ ]:
#generate pred using sir_eakf/sir_ah_eakf model
sir_eakf_model = "SIR-EAKF_edv_start_251004"
sir_eakf_start_date = pd.to_datetime('2025-10-04', format="%Y-%m-%d") #pd.to_datetime('2025-08-09', format="%Y-%m-%d") #pd.to_datetime('2025-09-06', format="%Y-%m-%d") #
for ref_date1 in ref_dates:
    dat_changerate_ref = df_hosp_ex[df_hosp_ex.date==ref_date1-timedelta(weeks=1)]
    # generate_sir_ah_eakf_pred(df_hosp_ex, sir_eakf_start_date, ref_date1, weeks_to_predict, 
    #                         locations, AH_daily, quantiles, num_samples, 
    #                         dat_changerate_ref, results_dir, model_desc=sir_eakf_model)
    generate_sir_eakf_pred(df_hosp_ex, sir_eakf_start_date, ref_date1, weeks_to_predict, 
                        locations, quantiles, num_samples, 
                        dat_changerate_ref, results_dir, model_desc=sir_eakf_model, 
                        random_state=100)

In [ ]:
#generate pred using seasonal drift model
epiweek_window = 1
pool_weight = 0 #0.5 #
seasonal_drift_model = "seasonal_drift" + model_suffix
if(pool_weight > 0) :
    seasonal_drift_model = seasonal_drift_model + f"_pool{pool_weight}"
print(seasonal_drift_model)
for ref_date1 in ref_dates:
    dat_changerate_ref = df_hosp_ex[df_hosp_ex.date==ref_date1-timedelta(weeks=1)]
    generate_seasonal_drift_pred(df_hosp_ex, ref_date1, weeks_to_predict, locations, quantiles, num_samples, epiweek_window,
                                dat_changerate_ref, results_dir, model_desc=seasonal_drift_model,
                                pool_weight=pool_weight, random_state=100)

In [ ]:
# models_to_plot = [sir_eakf_model] #[seasonal_drift_model] 
# calc_models_pred_fit(df_edv, models_to_plot, season, locations, 
#                      results_dir, figures_dir, alpha_vals, plot=True)

In [ ]:
# sel_lab_col = 'percent_positive'
# df_lab = read_lab_selected_data(data_dir, epiyear, epiweek, states, loc_name2abbr, sel_lab_col, plot=False)
# #add missing past values
# missing_dates = df_hosp_ex[~df_hosp_ex['date'].isin(df_lab['date'])]
# missing_dates_only = missing_dates[['date']].copy()
# for col in df_lab.columns:
#     if col != 'date':
#         missing_dates_only[col] = pd.NA
# df_lab = pd.concat([df_lab, missing_dates_only], ignore_index=True)
# df_lab = df_lab.sort_values(by='date').reset_index(drop=True)
# # To use as covariate we need to continue this forward up to max horizon
# for w in range(0,weeks_to_predict):
#     next_date = pd.Timestamp(ref_date) + timedelta(weeks=w)
#     df_lab.loc[len(df_lab)] = [next_date] + [pd.NA for s in range(num_states)]

# #Set to 0 values for states with missing lab for now
# df_lab['RI'] = 0 #pd.NA
# df_lab['PR'] = 0 #pd.NA
# df_lab['DC'] = 0 #pd.NA
# df_lab['NJ'] = 0 #pd.NA
# df_lab['NH'] = 0 #pd.NA
# df_lab['NV'] = 0 #pd.NA
# df_lab['AK'] = 0 #pd.NA
# df_lab['DE'] = 0 #pd.NA
# df_lab['UT'] = 0 #pd.NA

In [ ]:
# Run fit and predict darts models

df_past_covar = None
df_future_covar = None
# use_lab_covar = False
# use_ah_covar = False
# if(use_lab_covar):
#     df_past_covar = df_lab
# if(use_ah_covar):
#     df_future_covar = df_AH

nb_inflate = True

for model_desc in darts_local_models:
    print("-----------Model: {}-----------".format(model_desc))
    model = darts_local_models[model_desc]
    for ref_date1 in ref_dates:
        pred_start_date = ref_date1 
        model = model.untrained_model()
        dat_changerate_ref = df_hosp_ex[df_hosp_ex.date==ref_date1-timedelta(weeks=1)]

        df_prev_pred = load_pred_result_files(results_dir, model_desc, locations)
        if(df_prev_pred is not None):
            df_prev_pred = df_prev_pred[df_prev_pred['reference_date']<ref_date]
            df_prev_pred['location'] = df_prev_pred['location'].map(loc2abbr)

        pred = fit_and_predict_univariate(df_hosp_ex, states, model, model_desc, 
                                          ref_date1, weeks_to_predict, num_samples, 
                                          df_past_covar, df_future_covar,
                                          nb_inflate=nb_inflate,df_prev_pred=df_prev_pred)
        save_darts_pred_results_to_file(pred, ref_date1, weeks_to_predict, locations, quantiles,
                                        dat_changerate_ref, results_dir, model_desc)
        